# ColPali 第 4 路 round-trip 驗收（gate ③）— Colab runner

免裝環境、用 Colab 免費 GPU 跑 `scripts/colpali_roundtrip_check.py`，產出可驗的 `logs/roundtrip_result.json`（hit@5、per-query rank、sanity 控制）。

**跑之前先做兩件事：**
1. **開 GPU**：上方選單 `Runtime → Change runtime type → T4 GPU`。
2. **接受 PaliGemma 授權**：colpali-v1.2 依賴 *gated* `google/paligemma-3b-mix-448`。用你的 HF 帳號到 https://huggingface.co/google/paligemma-3b-mix-448 按一次同意，並備好一組 HF access token（https://huggingface.co/settings/tokens ，read 即可）。
3. **BQ 讀取權**：你的 Google 帳號需有 `polaris_core` 的 BigQuery 讀取權（讀 colpali_pages）。

依序執行每個 cell 即可。最後一個 cell 會下載結果，貼回 issue #17。


## 1. 取得程式（含合併好的 encoder + canonical runner）


In [ ]:
!git clone --depth 1 https://github.com/holajennytw/polaris-desk.git /content/polaris-desk
import os; os.chdir('/content/polaris-desk')
print('cwd =', os.getcwd())


## 2. 裝相依
專案鎖 Python 3.13，但 Colab 是 3.10/3.11 → 不跑 `pip install -e .`（會被 requires-python 擋）。改裝執行期相依，並把 `src/` 加進 PYTHONPATH（等同 repo 的 pytest pythonpath 設定）。


In [ ]:
!pip -q install colpali-engine google-cloud-bigquery pydantic-settings
# Colab 預裝 torchao 0.10.0 與 peft 不相容（LoRA 載入會 ImportError）。
# colpali-v1.2 不需要 torchao → 移掉，讓 peft 跳過量化探測。
!pip -q uninstall -y torchao
print('deps installed (torchao removed)')


## 3. 檢查 GPU（沒抓到就回上面開 T4）


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


## 4. HuggingFace 登入（gated paligemma）
執行後貼上你的 HF token（不會顯示）。


In [ ]:
from getpass import getpass
from huggingface_hub import login
import os
tok = getpass('HF access token: ').strip()
login(token=tok)
os.environ['HF_TOKEN'] = tok
print('HF logged in')


## 5. GCP 登入（讀 polaris_core.colpali_pages）
彈出 Google 授權，用**有 polaris_core 讀取權**的帳號。


In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCP authed')


## 6. Gold（repo 已內建重建版，可直接往下跑）

`data/gold/colpali_gold.json` 已隨 clone 下來（PM 2026-06-23 重建版，15 筆）。**想改用 R1 親挑版**就跑下面 cell 上傳覆蓋；不想換就直接跳到第 7 步。


In [ ]:
import os, json
# repo 已內建重建版 gold。要換成 R1 親挑版 → 把下行 FORCE_UPLOAD 設 True 再跑。
FORCE_UPLOAD = False
if FORCE_UPLOAD or not os.path.exists('data/gold/colpali_gold.json'):
    from google.colab import files; import shutil
    os.makedirs('data/gold', exist_ok=True)
    up = files.upload()  # 選 colpali_gold.json
    shutil.move(list(up)[0], 'data/gold/colpali_gold.json')
    print('已用上傳的 gold 覆蓋')
else:
    print('用 repo 內建 gold：data/gold/colpali_gold.json')
g = json.load(open('data/gold/colpali_gold.json'))
items = g.get('items', g) if isinstance(g, dict) else g
print('gold 筆數 =', len(items))


## 7. 跑 round-trip（canonical runner）
用 main 上合併好的 `ColpaliV12QueryEncoder`（**非**本地 fork 的 search_colpali），結果才可比。


In [ ]:
import os, subprocess, sys
env = {**os.environ,
       'PYTHONPATH': '/content/polaris-desk/src',
       'COLPALI_QUERY_ENCODER': '1',
       'VECTOR_BACKEND': 'bigquery',
       'GCP_PROJECT': 'polaris-desk-team',
       'BQ_DATASET': 'polaris_core'}
res = subprocess.run([sys.executable, 'scripts/colpali_roundtrip_check.py',
                      '--gold', 'data/gold/colpali_gold.json',
                      '--out', 'logs/roundtrip_result.json', '--verbose'],
                     cwd='/content/polaris-desk', env=env, text=True, capture_output=True)
print(res.stdout[-6000:])
if res.returncode != 0: print('--- STDERR ---', res.stderr[-3000:])


## 8. 結果 + 下載
看 `verdict` 與 `hit@5`。把下載的 `roundtrip_result.json` 附到 issue #17，並貼 metrics 表。


In [ ]:
import json
r = json.load(open('logs/roundtrip_result.json'))
print('VERDICT :', r['verdict'])
print('hit@5   :', r['metrics']['hit_at_5'])
print('hit@1   :', r['metrics']['hit_at_1'], '| hit@10:', r['metrics']['hit_at_10'], '| MRR:', r['metrics']['mrr'])
print('controls:', r['controls'])
from google.colab import files; files.download('logs/roundtrip_result.json')


## 9.（選用）把結果推回 repo
需要一組 GitHub PAT（scope `repo`）。沒有就跳過，改用上面下載的 JSON 附到 issue。


In [ ]:
from getpass import getpass
pat = getpass('GitHub PAT (repo scope), 留空跳過: ').strip()
if pat:
    !git config user.email 'r1@polaris.local' && git config user.name 'R1'
    !git checkout -b r4/colpali-gate3-result
    !git add data/gold/colpali_gold.json logs/roundtrip_result.json
    !git commit -m 'test(colpali): gate3 round-trip result'
    !git push https://{pat}@github.com/holajennytw/polaris-desk.git r4/colpali-gate3-result
    print('pushed branch r4/colpali-gate3-result — 開 PR 併入 main')
else:
    print('skipped — 請把下載的 roundtrip_result.json 附到 issue #17')
